# Section 4: Master Manifest Construction
Builds a strict dataset contract locking the operational dataset without overwriting the reconciled authoritative raw manifest.

In [1]:
import pandas as pd
import os
from IPython.display import display

OUTPUT_ROOT   = r"C:\SKIN CANCER v2\pipe output"
manifests_dir = os.path.join(OUTPUT_ROOT, "manifests")

raw_path   = os.path.join(manifests_dir, "authoritative_raw_manifest.csv")
clean_path = os.path.join(manifests_dir, "cleaned_working_manifest.csv")
train_path = os.path.join(manifests_dir, "training_eligible_manifest.csv")

## Load Manifests

In [2]:
print("Loading raw and cleaned manifests...")
df_raw   = pd.read_csv(raw_path)
df_clean = pd.read_csv(clean_path)
print(f"Raw manifest rows    : {len(df_raw):,}")
print(f"Cleaned manifest rows: {len(df_clean):,}")

Loading raw and cleaned manifests...
Raw manifest rows    : 20,720
Cleaned manifest rows: 20,720


## Normalize Schema

In [3]:
rename_map = {
    "matched_csv_image_id": "canonical_match_id",
    "review_status":        "reconciliation_status",
    "readable_status":      "file_accessible_status",
    "file_hash":            "duplicate_hash_group"
}
actual_renames = {k: v for k, v in rename_map.items() if k in df_clean.columns}
df_clean.rename(columns=actual_renames, inplace=True)

if "final_authoritative_label" not in df_clean.columns:
    df_clean["final_authoritative_label"] = df_clean["gt_label"]

legacy_cols = [
    "family_review_status", "family_review_decision",
    "remove_due_to_family_review_flag", "canonical_keep_flag"
]
df_clean.drop(columns=[c for c in legacy_cols if c in df_clean.columns], inplace=True)

if "final_dataset_status" in df_clean.columns:
    if "final_exclusion_reason" not in df_clean.columns:
        df_clean["final_exclusion_reason"] = "none"
    df_clean.loc[df_clean["final_dataset_status"] == "eligible", "final_exclusion_reason"] = "none"
    mask_excl = df_clean["final_dataset_status"].isin(["excluded", "flagged_for_review"])
    if mask_excl.any():
        known = {"duplicate_family_pending_review", "duplicate_family_permanently_excluded"}
        df_clean.loc[mask_excl & ~df_clean["final_exclusion_reason"].isin(known),
                     "final_exclusion_reason"] = "duplicate_family_pending_review"
print("Schema normalized.")

Schema normalized.


## Build Cleaned Manifest

In [4]:
df_clean.to_csv(clean_path, index=False)
print(f"Cleaned working manifest saved to {clean_path}.")
display(df_clean.head(2))

Cleaned working manifest saved to C:\SKIN CANCER v2\pipe output\manifests\cleaned_working_manifest.csv.


,full_path,folder_name,raw_folder_label,file_name_with_extension,file_stem_raw,base_id_candidate,recognized_suffix,extension,file_exists,file_size_bytes,...,reconciliation_status,review_decision,review_notes,duplicate_hash_group,family_id,final_dataset_status,final_exclusion_reason,eligible_for_training,eligible_for_split,final_authoritative_label
0,C:\SKIN CANCER v2\DS\NV\ISIC_0000000.jpg,NV,NV,ISIC_0000000.jpg,ISIC_0000000,ISIC_0000000,NaN,.jpg,True,49964,...,provisionally_approved,NaN,NaN,153b10f7b8b82bf80badbfdf73a544312637a1950e2dad...,FAM_ISIC_0000000,eligible,none,True,True,NV
1,C:\SKIN CANCER v2\DS\NV\ISIC_0000001.jpg,NV,NV,ISIC_0000001.jpg,ISIC_0000001,ISIC_0000001,NaN,.jpg,True,38941,...,provisionally_approved,NaN,NaN,6180745ca3044c6267b58dd77ae821fca7df549c64bb65...,FAM_ISIC_0000001,eligible,none,True,True,NV


## Build Training Eligible Manifest

In [5]:
required_final_cols = [
    "final_dataset_status", "final_exclusion_reason",
    "eligible_for_training", "eligible_for_split",
    "final_authoritative_label", "full_path"
]
missing_required = [c for c in required_final_cols if c not in df_clean.columns]
if missing_required:
    raise ValueError(f"Required columns missing: {missing_required}")

df_train = df_clean[
    (df_clean["final_dataset_status"] == "eligible") &
    (df_clean["eligible_for_training"] == True) &
    (df_clean["eligible_for_split"] == True)
].copy()

df_train.to_csv(train_path, index=False)
print(f"Training eligible manifest saved to {train_path}.")
print(f"Training eligible row count: {len(df_train):,}")
display(df_train.head(5))

Training eligible manifest saved to C:\SKIN CANCER v2\pipe output\manifests\training_eligible_manifest.csv.
Training eligible row count: 20,664


,full_path,folder_name,raw_folder_label,file_name_with_extension,file_stem_raw,base_id_candidate,recognized_suffix,extension,file_exists,file_size_bytes,...,reconciliation_status,review_decision,review_notes,duplicate_hash_group,family_id,final_dataset_status,final_exclusion_reason,eligible_for_training,eligible_for_split,final_authoritative_label
0,C:\SKIN CANCER v2\DS\NV\ISIC_0000000.jpg,NV,NV,ISIC_0000000.jpg,ISIC_0000000,ISIC_0000000,NaN,.jpg,True,49964,...,provisionally_approved,NaN,NaN,153b10f7b8b82bf80badbfdf73a544312637a1950e2dad...,FAM_ISIC_0000000,eligible,none,True,True,NV
1,C:\SKIN CANCER v2\DS\NV\ISIC_0000001.jpg,NV,NV,ISIC_0000001.jpg,ISIC_0000001,ISIC_0000001,NaN,.jpg,True,38941,...,provisionally_approved,NaN,NaN,6180745ca3044c6267b58dd77ae821fca7df549c64bb65...,FAM_ISIC_0000001,eligible,none,True,True,NV
2,C:\SKIN CANCER v2\DS\NV\ISIC_0000003.jpg,NV,NV,ISIC_0000003.jpg,ISIC_0000003,ISIC_0000003,NaN,.jpg,True,45774,...,provisionally_approved,NaN,NaN,e3092bd47d1955c117a18acb1e236222cae99acfbd393d...,FAM_ISIC_0000003,eligible,none,True,True,NV
3,C:\SKIN CANCER v2\DS\NV\ISIC_0000006.jpg,NV,NV,ISIC_0000006.jpg,ISIC_0000006,ISIC_0000006,NaN,.jpg,True,58570,...,provisionally_approved,NaN,NaN,755896e5f1593d8b705f3c54c9b2632c7ba894eb89c362...,FAM_ISIC_0000006,eligible,none,True,True,NV
4,C:\SKIN CANCER v2\DS\NV\ISIC_0000007.jpg,NV,NV,ISIC_0000007.jpg,ISIC_0000007,ISIC_0000007,NaN,.jpg,True,54175,...,provisionally_approved,NaN,NaN,2d3fe1d91afb6dcf7f21893ec8bfaf42b8ed885026bf88...,FAM_ISIC_0000007,eligible,none,True,True,NV


## Consistency Validations

In [6]:
print("=== CONSISTENCY VALIDATION ===")
check_1 = all(df_train["full_path"].isin(df_clean["full_path"]))
print(f"1. All training rows exist in cleaned  : {check_1}")
check_2 = all(df_clean["full_path"].isin(df_raw["full_path"]))
print(f"2. All cleaned rows exist in raw       : {check_2}")
check_3 = len(df_train[df_train["final_dataset_status"] == "excluded"]) == 0
print(f"3. No excluded row in training manifest: {check_3}")
check_4 = all(df_train["final_exclusion_reason"] == "none")
print(f"4. All training rows have reason=none  : {check_4}")

eligible_count = len(df_train)
excluded_count = len(df_clean[~df_clean["eligible_for_training"].fillna(False).astype(bool)])
class_counts   = df_train["final_authoritative_label"].value_counts()
print(f"\nEligible row count : {eligible_count:,}")
print(f"Excluded/flagged   : {excluded_count:,}")
print(f"Class counts:\n{class_counts.to_string()}")

=== CONSISTENCY VALIDATION ===
1. All training rows exist in cleaned  : True
2. All cleaned rows exist in raw       : True
3. No excluded row in training manifest: True
4. All training rows have reason=none  : True

Eligible row count : 20,664
Excluded/flagged   : 56
Class counts:
final_authoritative_label
NV     12851
MEL     4504
BCC     3309


## Save Summary & Schema Artifacts

In [7]:
report_data = {
    "total_reconciled_rows": len(df_clean),
    "total_eligible_rows":   eligible_count,
    "total_excluded_rows":   excluded_count,
    "eligible_NV_count":     int(class_counts.get("NV", 0)),
    "eligible_MEL_count":    int(class_counts.get("MEL", 0)),
    "eligible_BCC_count":    int(class_counts.get("BCC", 0))
}
report_path = os.path.join(manifests_dir, "manifest_summary_report.csv")
pd.DataFrame([report_data]).to_csv(report_path, index=False)

schema_text = (
    "Manifest Schema (v2)\n\n"
    "full_path                  : Absolute file path.\n"
    "folder_name                : Folder where discovered (NV, MEL, BCC).\n"
    "file_name_with_extension   : Original filename.\n"
    "file_stem_raw              : Filename without extension.\n"
    "extension                  : File extension.\n"
    "base_id_candidate          : Base ID after suffix normalization.\n"
    "recognized_suffix          : Detected suffix e.g. _downsampled.\n"
    "canonical_match_id         : ID matched to ground truth CSV.\n"
    "folder_label               : Provisional label from folder.\n"
    "gt_label                   : Authoritative label from GT CSV.\n"
    "final_authoritative_label  : Final label for pipeline (NV/MEL/BCC).\n"
    "match_status               : matched_exact | matched_via_suffix_rule | unmatched.\n"
    "label_agreement_status     : agree | disagree | missing_ground_truth.\n"
    "age_approx                 : Audit/split-support only. Not model input.\n"
    "sex                        : Audit/split-support only. Not model input.\n"
    "anatom_site_general        : Audit/split-support only. Not model input.\n"
    "lesion_id                  : Split-support only. Not model input.\n"
    "metadata_row_found         : Boolean metadata match.\n"
    "family_id                  : Duplicate-family group identifier.\n"
    "duplicate_hash_group       : Hash-based exact duplicate group.\n"
    "final_dataset_status       : eligible | flagged_for_review | excluded.\n"
    "final_exclusion_reason     : none | duplicate_family_pending_review.\n"
    "eligible_for_training      : Boolean.\n"
    "eligible_for_split         : Boolean.\n\n"
    "Rule: downstream stages must use training_eligible_manifest.csv.\n"
    "Do not re-infer dataset state by rescanning folders.\n"
)
schema_path = os.path.join(manifests_dir, "manifest_schema.txt")
with open(schema_path, "w", encoding="utf-8") as f:
    f.write(schema_text)
print(f"Summary report : {report_path}")
print(f"Schema file    : {schema_path}")

Summary report : C:\SKIN CANCER v2\pipe output\manifests\manifest_summary_report.csv
Schema file    : C:\SKIN CANCER v2\pipe output\manifests\manifest_schema.txt


In [8]:
# Final summary
print("=" * 60)
print("  04_master_manifest -- FINAL SUMMARY")
print("=" * 60)
print(f"\nTotal reconciled rows     : {len(df_clean):,}")
print(f"Eligible rows             : {eligible_count:,}")
print(f"Excluded/flagged rows     : {excluded_count:,}")
print(f"\nClass counts (eligible):")
for label in ["NV", "MEL", "BCC"]:
    print(f"  {label}: {int(class_counts.get(label, 0)):,}")
print(f"\nFinal manifest paths:")
print(f"  Cleaned : {clean_path}")
print(f"  Training: {train_path}")
print(f"\nOutput file verification:")
for p in [raw_path, clean_path, train_path, report_path, schema_path]:
    exists = os.path.exists(p)
    size   = os.path.getsize(p) if exists else 0
    status = "OK" if exists else "MISSING"
    print(f"  [{status}] {os.path.basename(p):<45} {size:>10,} bytes")
print("=" * 60)

  04_master_manifest -- FINAL SUMMARY

Total reconciled rows     : 20,720
Eligible rows             : 20,664
Excluded/flagged rows     : 56

Class counts (eligible):
  NV: 12,851
  MEL: 4,504
  BCC: 3,309

Final manifest paths:
  Cleaned : C:\SKIN CANCER v2\pipe output\manifests\cleaned_working_manifest.csv
  Training: C:\SKIN CANCER v2\pipe output\manifests\training_eligible_manifest.csv

Output file verification:
  [OK] authoritative_raw_manifest.csv                 4,657,009 bytes
  [OK] cleaned_working_manifest.csv                   6,925,666 bytes
  [OK] training_eligible_manifest.csv                 6,905,055 bytes
  [OK] manifest_summary_report.csv                          151 bytes
  [OK] manifest_schema.txt                                1,648 bytes
